# EEG Project Sample Code: Data Loading, Retrieval Evaluation, and Reconstruction Evaluation

This notebook provides concise starter code for the course project. It demonstrates:

1. how to load the released EEG data from `train.pt` and `test.pt`,
2. how to evaluate **200-way zero-shot retrieval** with **Top-1** and **Top-5** accuracy, and
3. how to evaluate **image reconstruction** with the official TA evaluation function.

The project specification requires retrieval evaluation with **Top-1 Accuracy** and **Top-5 Accuracy**, and reconstruction evaluation with the official TA code including metrics such as **SSIM** and **CLIP-based evaluation**. The report should summarize results over **10 different random seeds** using **mean ± standard deviation**. 

## 1. Environment and Conventions

This notebook is written for a simple teaching setting:

- Only **EEG** signals are considered.
- A single subject is fixed by the released files `train.pt` and `test.pt`.
- Therefore, the loader does **not** expose `brain_key` or `subject_id` arguments.
- The core input is a tensor of shape **`[N, C, T]`**, where:
  - `N` is the number of samples,
  - `C` is the number of EEG channels,
  - `T` is the number of time points.

This is clearly a **multichannel time-series learning** problem.

For image pairing, the dataset provides an `image_id` for each EEG sample. Once `image_id` is known, the paired image name is also known. For example, identifiers such as `aardvark_01b` and `aardvark_04s` indicate the corresponding image stems. The image folder can be resolved from the identifier convention used in the released data, so **image loading is intentionally omitted in this notebook**.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import List, Literal, Optional, Sequence, Union

import datasets
import numpy as np
import torch

## 2. EEG Dataset Loader

The function below is a simplified version of the original helper:

- `brain_key` is removed, because students only need to handle **EEG**.
- `subject_ids` is removed, because the course project fixes one subject through the provided `train.pt` and `test.pt`.
- All dataset descriptions are kept generic and do not depend on any external project name.
- `selected_channels` is still supported, so students can work either with **all channels** or with a **chosen subset**.

In [ ]:
def _selected_channel_indices_from_jsonl(
    selected_channels: Union[str, Sequence[str]],
    eeg_channel_jsonl: Union[str, Path],
) -> List[int]:
    """Map EEG channel names to channel indices.

    The JSONL file is expected to store channel metadata line by line.
    Each line may contain fields such as:
    - {"name": "F5"}
    - {"channel_name": "F5"}
    - {"label": "F5"}

    Parameters
    ----------
    selected_channels:
        A single channel name or a sequence of channel names.
    eeg_channel_jsonl:
        Path to the JSONL file that defines the EEG channel order.

    Returns
    -------
    List[int]
        The indices of the requested channels in the original EEG tensor.
    """
    if isinstance(selected_channels, str):
        selected_channels = [selected_channels]
    selected_channels = list(selected_channels)

    channel_names: List[str] = []
    with open(eeg_channel_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            name = item.get("name") or item.get("channel_name") or item.get("label")
            if name is None:
                raise KeyError(
                    "Each JSONL record must contain one of: 'name', 'channel_name', or 'label'."
                )
            channel_names.append(str(name))

    name_to_index = {name: idx for idx, name in enumerate(channel_names)}

    missing = [ch for ch in selected_channels if ch not in name_to_index]
    if missing:
        raise ValueError(f"Unknown EEG channels: {missing}")

    return [name_to_index[ch] for ch in selected_channels]


def load_eeg_dataset(
    *,
    data_directory: Union[str, Path],
    split: Literal["train", "test"],
    avg_trials: bool = True,
    selected_channels: Optional[Union[str, Sequence[str]]] = None,
    eeg_channel_jsonl: Union[str, Path] = "image-eeg-data/THINGS_EEG_CHANNELS.jsonl",
) -> datasets.Dataset:
    """Build a Hugging Face dataset for the released EEG data.

    The expected file structure is:

        data_directory/
        ├── train.pt
        └── test.pt
        └── THINGS_EEG_CHANNELS.jsonl

    The returned dataset contains:
    - `eeg`: Array2D float32 with shape [C, T]
    - `image_id`: string

    Parameters
    ----------
    data_directory:
        Directory that stores `train.pt` and `test.pt`.
    split:
        Either `"train"` or `"test"`.
    avg_trials:
        If the loaded EEG tensor has shape `[N, TRIAL, C, T]`, average over the
        trial dimension and return `[N, C, T]`. If `False`, flatten trials into
        the sample dimension and return `[(N * TRIAL), C, T]`.
    selected_channels:
        Optional EEG channel subset. If provided, the loader keeps only the
        requested channels in the given order.
    eeg_channel_jsonl:
        JSONL file describing the EEG channel order.

    Returns
    -------
    datasets.Dataset
        A dataset with columns:
        - `eeg`: Array2D float32 [C, T]
        - `image_id`: string
    """
    pt_path = Path(data_directory).joinpath(f"{split}.pt")
    loaded = torch.load(str(pt_path), weights_only=False)

    x = torch.as_tensor(loaded["eeg"])  # [N, TRIAL, C, T] or [N, C, T]
    if x.ndim == 4:
        if avg_trials:
            x = x.mean(dim=1)  # [N, C, T]
        else:
            x = x.reshape(-1, *x.shape[2:])  # [N * TRIAL, C, T]
    elif x.ndim != 3:
        raise ValueError(f"Unexpected EEG shape: {tuple(x.shape)} in {pt_path}")

    if selected_channels is not None:
        sel_idx = _selected_channel_indices_from_jsonl(selected_channels, eeg_channel_jsonl)
        x = x[:, sel_idx, :]

    imgs = np.array(loaded["img"])
    if avg_trials:
        if imgs.ndim == 2:
            imgs = imgs[:, 0]
        imgs = imgs.reshape(-1)[: x.shape[0]]
    else:
        imgs = imgs.reshape(-1)

    image_ids = [Path(p).stem for p in imgs.tolist()]
    if len(image_ids) != x.shape[0]:
        raise ValueError(
            f"EEG/image mismatch: {x.shape[0]} vs {len(image_ids)} for {pt_path}"
        )

    x_np = x.float().cpu().numpy()  # [N, C, T]
    C, T = x_np.shape[1], x_np.shape[2]

    features = datasets.Features(
        {
            "eeg": datasets.Array2D(shape=(C, T), dtype="float32"),
            "image_id": datasets.Value("string"),
        }
    )

    ds = datasets.Dataset.from_dict(
        {
            "eeg": list(x_np),
            "image_id": image_ids,
        },
        features=features,
    )
    return ds

### 2.1 Example: load all EEG channels

This version keeps the full multichannel EEG time series.

In [ ]:
train_ds_all = load_eeg_dataset(
    data_directory="image-eeg-data",
    split="train",
    avg_trials=True,
)

train_ds_all = train_ds_all.with_format("torch")

test_ds_all = load_eeg_dataset(
    data_directory="image-eeg-data",
    split="test",
    avg_trials=True,
)

test_ds_all = test_ds_all.with_format("torch")
print(train_ds_all)
print(test_ds_all)
print(train_ds_all[0]["eeg"].shape)  # expected: [C, T]
print(train_ds_all[0]["image_id"])

### 2.2 Example: load a selected channel subset

This version keeps only four EEG channels: `["F5", "FC5", "FT10", "T7"]`.

Because the input remains an ordered signal over time, this still forms a very clear **time-series modeling** problem, now with a smaller channel dimension.

In [ ]:
train_ds_subset = load_eeg_dataset(
    data_directory="image-eeg-data",
    split="train",
    avg_trials=True,
    selected_channels=["F5", "FC5", "FT10", "T7"],
    eeg_channel_jsonl="image-eeg-data/THINGS_EEG_CHANNELS.jsonl",
)

train_ds_subset = train_ds_subset.with_format("torch")

print(train_ds_subset)
print(train_ds_subset[0]["eeg"].shape)  # expected: [4, T]

## 3. Retrieval Evaluation Example

The project requires a **200-way zero-shot retrieval** evaluation with:

- **Top-1 Accuracy**
- **Top-5 Accuracy**

Below we generate random brain embeddings and random image embeddings for demonstration only. We then compute a similarity matrix and evaluate retrieval under the standard setting in which each brain embedding should match the image embedding with the same index.

For reporting, the trained model should be evaluated under **10 different random seeds**, and the final report should present **mean ± standard deviation**.

In [ ]:
import random
from typing import Dict

import numpy as np
import torch
import torch.nn.functional as F

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_retrieval_metrics(logits: torch.Tensor) -> Dict[str, float]:
    """Compute Top-1 and Top-5 retrieval accuracy.

    Parameters
    ----------
    logits:
        Similarity matrix of shape [N, N], where row i corresponds to the
        i-th EEG sample and column j corresponds to the j-th candidate image.

    Returns
    -------
    Dict[str, float]
        A dictionary with Top-1 and Top-5 accuracy.
    """
    if logits.ndim != 2 or logits.shape[0] != logits.shape[1]:
        raise ValueError("Expected a square similarity matrix of shape [N, N].")

    n = logits.shape[0]
    targets = torch.arange(n)

    top1_pred = logits.argmax(dim=1)
    top1_acc = (top1_pred == targets).float().mean().item()

    top5_idx = logits.topk(k=5, dim=1).indices
    top5_acc = (top5_idx == targets[:, None]).any(dim=1).float().mean().item()

    return {
        "top1_acc": top1_acc,
        "top5_acc": top5_acc,
    }


def demo_retrieval_one_seed(
    seed: int,
    num_items: int = 200,
    embedding_dim: int = 512,
) -> Dict[str, float]:
    """Generate random embeddings and evaluate 200-way retrieval."""
    set_seed(seed)

    brain_embeddings = torch.randn(num_items, embedding_dim)
    image_embeddings = torch.randn(num_items, embedding_dim)

    brain_embeddings = F.normalize(brain_embeddings, dim=1)
    image_embeddings = F.normalize(image_embeddings, dim=1)

    logits = brain_embeddings @ image_embeddings.T
    metrics = compute_retrieval_metrics(logits)
    return metrics


def summarize_metrics_over_seeds(metric_list):
    keys = metric_list[0].keys()
    summary = {}
    for key in keys:
        values = np.array([m[key] for m in metric_list], dtype=np.float64)
        summary[key] = {
            "mean": float(values.mean()),
            "std": float(values.std(ddof=1)),
        }
    return summary

In [ ]:
# Demonstration: 200-way zero-shot retrieval over 10 seeds
retrieval_results = []
for seed in range(10):
    metrics = demo_retrieval_one_seed(seed=seed, num_items=200, embedding_dim=512)
    retrieval_results.append(metrics)

retrieval_summary = summarize_metrics_over_seeds(retrieval_results)

print("Per-seed retrieval metrics:")
for seed, metrics in enumerate(retrieval_results):
    print(
        f"seed={seed:02d} | "
        f"Top-1={metrics['top1_acc']:.4f} | "
        f"Top-5={metrics['top5_acc']:.4f}"
    )

print("\nSummary over 10 seeds:")
for key, stats in retrieval_summary.items():
    print(f"{key}: {stats['mean']:.4f} ± {stats['std']:.4f}")

## 4. Reconstruction Evaluation Example

The official reconstruction evaluation should be performed with the TA evaluation script. The uploaded `utils_eval.py` contains the full implementation of:

- pixel correlation,
- SSIM,
- AlexNet-based perceptual identification,
- Inception-based identification,
- CLIP-based identification,
- EfficientNet correlation,
- SwAV correlation,
- and the final `eval_images(...)` wrapper.

Below, we inline that code into the notebook so that students can call the evaluation directly.

### Important note

This evaluation code uses pretrained vision models. On first execution, PyTorch or related packages may need to download pretrained weights. The code below is included as the **official evaluation example**, but it can be computationally expensive.

In [ ]:
# Install required dependencies for evaluation
!pip install -q scikit-image
!pip install -q git+https://github.com/openai/CLIP.git

In [ ]:
import numpy as np
import torch
from torchvision import transforms
from tqdm import tqdm
from torchvision.models.feature_extraction import create_feature_extractor
from skimage.color import rgb2gray
from skimage.metrics import structural_similarity
from torchvision.models import inception_v3, Inception_V3_Weights
import clip
from torchvision.models import efficientnet_b1, EfficientNet_B1_Weights
import scipy as sp
import os
from PIL import Image
import datetime
import json
import glob


@torch.no_grad()
def two_way_identification(
    all_brain_recons,
    all_images,
    model,
    preprocess,
    feature_layer=None,
    return_avg=True,
    device: torch.device = torch.device("cpu"),
):
    preds = model(
        torch.stack([preprocess(recon) for recon in all_brain_recons], dim=0).to(device)
    )
    reals = model(
        torch.stack([preprocess(indiv) for indiv in all_images], dim=0).to(device)
    )
    if feature_layer is None:
        preds = preds.float().flatten(1).cpu().numpy()
        reals = reals.float().flatten(1).cpu().numpy()
    else:
        preds = preds[feature_layer].float().flatten(1).cpu().numpy()
        reals = reals[feature_layer].float().flatten(1).cpu().numpy()

    r = np.corrcoef(reals, preds)
    r = r[: len(all_images), len(all_images) :]
    congruents = np.diag(r)

    success = r < congruents
    success_cnt = np.sum(success, 0)

    if return_avg:
        perf = np.mean(success_cnt) / (len(all_images) - 1)
        return perf
    else:
        return success_cnt, len(all_images) - 1


def pixcorr(all_images, all_brain_recons):
    preprocess = transforms.Compose(
        [
            transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR),
        ]
    )

    all_images_flattened = preprocess(all_images).reshape(len(all_images), -1).cpu()
    all_brain_recons_flattened = (
        preprocess(all_brain_recons).reshape(len(all_brain_recons), -1).cpu()
    )

    corrsum = 0
    n = min(len(all_images_flattened), len(all_brain_recons_flattened))
    for i in tqdm(range(n)):
        corrsum += np.corrcoef(all_images_flattened[i], all_brain_recons_flattened[i])[
            0
        ][1]
    corrmean = corrsum / n

    pixcorr = corrmean
    return pixcorr


def ssim(all_images, all_brain_recons):
    preprocess = transforms.Compose(
        [
            transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR),
        ]
    )

    img_gray = rgb2gray(preprocess(all_images).permute((0, 2, 3, 1)).cpu())
    recon_gray = rgb2gray(preprocess(all_brain_recons).permute((0, 2, 3, 1)).cpu())

    ssim_score = []
    for im, rec in tqdm(zip(img_gray, recon_gray), total=len(all_images)):
        ssim_score.append(
            structural_similarity(
                rec,
                im,
                multichannel=True,
                gaussian_weights=True,
                sigma=1.5,
                use_sample_covariance=False,
                data_range=1.0,
            )
        )

    ssim = np.mean(ssim_score)
    return ssim


def alexnet(all_images, all_brain_recons, device: torch.device = torch.device("cpu")):
    from torchvision.models import alexnet, AlexNet_Weights

    alex_weights = AlexNet_Weights.IMAGENET1K_V1

    alex_model = create_feature_extractor(
        alexnet(weights=alex_weights), return_nodes=["features.4", "features.11"]
    ).to(device)
    alex_model.eval().requires_grad_(False)

    preprocess = transforms.Compose(
        [
            transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    all_per_correct = two_way_identification(
        all_brain_recons.float(),
        all_images,
        alex_model,
        preprocess,
        "features.4",
        device=device,
    )
    alexnet2 = np.mean(all_per_correct)

    all_per_correct = two_way_identification(
        all_brain_recons.float(),
        all_images,
        alex_model,
        preprocess,
        "features.11",
        device=device,
    )
    alexnet5 = np.mean(all_per_correct)
    return alexnet2, alexnet5


def inception(all_images, all_brain_recons, device: torch.device = torch.device("cpu")):
    weights = Inception_V3_Weights.DEFAULT
    inception_model = create_feature_extractor(
        inception_v3(weights=weights), return_nodes=["avgpool"]
    ).to(device)
    inception_model.eval().requires_grad_(False)

    preprocess = transforms.Compose(
        [
            transforms.Resize(342, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    all_per_correct = two_way_identification(
        all_brain_recons,
        all_images,
        inception_model,
        preprocess,
        "avgpool",
        device=device,
    )

    inception = np.mean(all_per_correct)
    return inception


def clip_(all_images, all_brain_recons, device: torch.device = torch.device("cpu")):
    clip_model, preprocess = clip.load("ViT-L/14", device=device)

    preprocess = transforms.Compose(
        [
            transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.Normalize(
                mean=[0.48145466, 0.4578275, 0.40821073],
                std=[0.26862954, 0.26130258, 0.27577711],
            ),
        ]
    )

    all_per_correct = two_way_identification(
        all_brain_recons,
        all_images,
        clip_model.encode_image,
        preprocess,
        None,
        device=device,
    )
    clip_ = np.mean(all_per_correct)
    return clip_


def effnet(all_images, all_brain_recons, device: torch.device = torch.device("cpu")):
    weights = EfficientNet_B1_Weights.DEFAULT
    eff_model = create_feature_extractor(
        efficientnet_b1(weights=weights), return_nodes=["avgpool"]
    ).to(device)
    eff_model.eval().requires_grad_(False)

    preprocess = transforms.Compose(
        [
            transforms.Resize(255, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    gt = eff_model(preprocess(all_images))["avgpool"]
    gt = gt.reshape(len(gt), -1).cpu().numpy()
    fake = eff_model(preprocess(all_brain_recons))["avgpool"]
    fake = fake.reshape(len(fake), -1).cpu().numpy()

    effnet = np.array(
        [sp.spatial.distance.correlation(gt[i], fake[i]) for i in range(len(gt))]
    ).mean()
    return effnet


def swav(all_images, all_brain_recons, device: torch.device = torch.device("cpu")):
    swav_model = torch.hub.load("facebookresearch/swav:main", "resnet50")
    swav_model = create_feature_extractor(swav_model, return_nodes=["avgpool"]).to(
        device
    )
    swav_model.eval().requires_grad_(False)

    preprocess = transforms.Compose(
        [
            transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    gt = swav_model(preprocess(all_images))["avgpool"]
    gt = gt.reshape(len(gt), -1).cpu().numpy()
    fake = swav_model(preprocess(all_brain_recons))["avgpool"]
    fake = fake.reshape(len(fake), -1).cpu().numpy()

    swav = np.array(
        [sp.spatial.distance.correlation(gt[i], fake[i]) for i in range(len(gt))]
    ).mean()
    return swav


def eval_images(
    real_images: torch.Tensor,
    fake_images: torch.Tensor,
    device: torch.device = torch.device("cpu"),
):
    real_images = real_images.to(device).float()
    fake_images = fake_images.to(device).float()

    pixcorrs = pixcorr(real_images, fake_images)
    ssims = ssim(real_images, fake_images)
    alex2, alex5 = alexnet(real_images, fake_images, device=device)
    inceptions = inception(real_images, fake_images, device=device)
    clips = clip_(real_images, fake_images, device=device)
    effnets = effnet(real_images, fake_images, device=device)
    swavs = swav(real_images, fake_images, device=device)

    return {
        "eval_pixcorr": pixcorrs.item(),
        "eval_ssim": ssims.item(),
        "eval_alex2": alex2.item(),
        "eval_alex5": alex5.item(),
        "eval_inception": inceptions.item(),
        "eval_clip": clips.item(),
        "eval_effnet": effnets.item(),
        "eval_swav": swavs.item(),
    }

### 4.1 Random reconstruction example

For demonstration, we generate:

- `200` reference images, and
- `200 × 10` random reconstructed images,

which matches the instruction that each image is repeated **10 times**. A simple evaluation protocol is to compute the official metrics independently for each seed and then report **mean ± standard deviation** over 10 runs.

The code below generates random tensors with shape `[200, 3, 256, 256]` for each run and evaluates them with `eval_images(...)`.

In [ ]:
def demo_reconstruction_one_seed(
    seed: int,
    num_images: int = 200,
    image_size: int = 256,
    device: str = "cpu",
):
    set_seed(seed)

    # Random images in [0, 1]
    real_images = torch.rand(num_images, 3, image_size, image_size)
    fake_images = torch.rand(num_images, 3, image_size, image_size)

    metrics = eval_images(
        real_images=real_images,
        fake_images=fake_images,
        device=torch.device(device),
    )
    return metrics

In [ ]:
# Official-style summary over 10 seeds
# Note: this cell can be slow because eval_images uses multiple pretrained models.
reconstruction_results = []
for seed in range(10):
    metrics = demo_reconstruction_one_seed(
        seed=seed,
        num_images=200,
        image_size=256,
        device="cpu",  # change to "cuda" if available
    )
    reconstruction_results.append(metrics)

reconstruction_summary = summarize_metrics_over_seeds(reconstruction_results)

print("Per-seed reconstruction metrics:")
for seed, metrics in enumerate(reconstruction_results):
    print(f"seed={seed:02d} | {metrics}")

print("\nSummary over 10 seeds:")
for key, stats in reconstruction_summary.items():
    print(f"{key}: {stats['mean']:.4f} ± {stats['std']:.4f}")

## 5. Reporting Template

In the project report, students should present results in the following form:

### Retrieval
- Top-1 Accuracy: `mean ± std` over 10 seeds
- Top-5 Accuracy: `mean ± std` over 10 seeds

### Reconstruction
- Official evaluation metrics from `eval_images(...)`
- At minimum, clearly report:
  - `eval_ssim`
  - `eval_clip`
- Report all metrics as `mean ± std` over 10 seeds whenever the protocol requires repeated runs.

This notebook is intended as a **minimal and readable reference implementation** for data loading and evaluation only. Model design, training, and inference can be built on top of it.